In [1]:
import os
import re
import unidecode
from pyspark.sql import SparkSession
import pyspark.sql.functions as sf
from pyspark.sql.types import StructType, StructField, StringType, FloatType, TimestampType

# --- Função de Processamento para cada Arquivo ---
# Esta função será distribuída e executada em paralelo pelo Spark para cada arquivo.
def processar_conteudo_arquivo(file_info):
    """
    Processa o conteúdo de um único arquivo CSV, extraindo metadados do cabeçalho
    e os dados principais.
    
    Args:
        file_info (tuple): Uma tupla contendo (caminho_do_arquivo, conteudo_completo_do_arquivo).

    Returns:
        list: Uma lista de dicionários, onde cada dicionário representa uma linha de dados
              enriquecida com os metadados do arquivo.
    """
    caminho_arquivo, conteudo_completo = file_info
    linhas = conteudo_completo.split('\n')
    
    # 1. Extração de Metadados do Cabeçalho (primeiras 8 linhas)
    metadados = {}
    cabecalho_linhas = linhas[:8]
    for linha in cabecalho_linhas:
        if ';' in linha:
            partes = linha.split(';', 1)
            chave = partes[0].strip()
            valor = partes[1].strip() if len(partes) > 1 else None
            
            # Limpeza e normalização da chave
            chave_limpa = unidecode.unidecode(chave).upper()
            chave_limpa = re.sub(r'[^A-Z0-9 ]', '', chave_limpa)
            if 'REGIO' in chave_limpa: chave_limpa = 'REGIAO'
            if 'ESTACO' in chave_limpa: chave_limpa = 'ESTACAO'
            
            metadados[chave_limpa] = valor

    # 2. Extração de Dados Tabulares (a partir da linha 9)
    registros = []
    dados_linhas = linhas[9:] # A linha 8 é o header dos dados
    
    for linha_dado in dados_linhas:
        valores = linha_dado.strip().split(';')
        # Garante que a linha tem o número mínimo de colunas para evitar erros
        if len(valores) >= 11 and valores[0]: # Aumentado para 11 para garantir a coluna de temperatura
            try:
                # O nome da coluna de data pode variar
                data = valores[0]
                hora = valores[1]
                # O nome da coluna de temperatura é mais consistente
                temperatura = valores[10] # Baseado no formato de dados do INMET
                
                # Monta um dicionário para a linha de dados
                registro = {
                    'Regiao': metadados.get('REGIAO'),
                    'UF': metadados.get('UF'),
                    'Estacao': metadados.get('ESTACAO'),
                    'Latitude': float(str(metadados.get('LATITUDE', '0')).replace(',', '.')),
                    'Longitude': float(str(metadados.get('LONGITUDE', '0')).replace(',', '.')),
                    'Altitude': float(str(metadados.get('ALTITUDE', '0')).replace(',', '.')),
                    'DataOriginal': data,
                    'HoraOriginal': hora,
                    'TemperaturaOriginal': temperatura
                }
                registros.append(registro)
            except (IndexError, ValueError) as e:
                # Ignora linhas malformadas, opcionalmente pode-se logar o erro
                # print(f"Linha ignorada por erro: {e} -> {linha_dado}")
                pass
                
    return registros

# --- Script Principal PySpark ---
def main():
    spark = SparkSession.builder \
        .appName("Processamento INMET com PySpark") \
        .master("spark://spark-master:7077") \
        .getOrCreate()

    try:
        # Ajuste o caminho para ser mais flexível, se necessário
        caminho_base = "../data/bronze/INMET/*/*.CSV"
        pasta_destino = "../data/bronze/INMET_PARQUET"
        caminho_leitura = f"{caminho_base}" # Lê de todos os anos

        rdd_arquivos = spark.sparkContext.wholeTextFiles(caminho_leitura)
        rdd_processado = rdd_arquivos.flatMap(processar_conteudo_arquivo)
        
        # Persist para evitar recomputação
        rdd_processado.persist()

        if rdd_processado.isEmpty():
            print("Nenhum dado foi processado. Verifique o caminho e o conteúdo dos arquivos.")
            return

        # --- SCHEMA EXPLÍCITO ---
        schema_definido = StructType([
            StructField("Regiao", StringType(), True),
            StructField("UF", StringType(), True),
            StructField("Estacao", StringType(), True),
            StructField("Latitude", FloatType(), True),
            StructField("Longitude", FloatType(), True),
            StructField("Altitude", FloatType(), True),
            StructField("DataOriginal", StringType(), True),
            StructField("HoraOriginal", StringType(), True),
            StructField("TemperaturaOriginal", StringType(), True)
        ])

        # Cria o DataFrame fornecendo o schema para evitar inferência.
        df = spark.createDataFrame(rdd_processado, schema=schema_definido)

        df.show(10,truncate=False)

        df_transformado = (
            df
            .withColumn('Temperatura', sf.regexp_replace(sf.col('TemperaturaOriginal'), ',', '.').try_cast(FloatType()))
            .withColumn("DataHoraOriginal", sf.concat_ws(" ", sf.col("DataOriginal"), sf.col("HoraOriginal")))
            .withColumn('_data_timestamp',sf.coalesce(
                sf.try_to_timestamp(sf.col("DataHoraOriginal") , sf.lit('yyyy-MM-dd HHmm')),
                sf.try_to_timestamp(sf.col("DataHoraOriginal") , sf.lit('yyyy-MM-dd HH:mm')),
                sf.try_to_timestamp(sf.col("DataHoraOriginal") , sf.lit('yyyy/MM/dd HHmm')),
                sf.try_to_timestamp(sf.col("DataHoraOriginal") , sf.lit('yyyy/MM/dd HH:mm'))
                        )
                       )
            .filter(sf.col('_data_timestamp').isNotNull())
            .filter(sf.col('Temperatura') != -9999.0)
            .withColumn('Data', sf.date_format(sf.col('_data_timestamp'), 'yyyy-MM-dd'))
            .withColumn('Tempo', sf.date_format(sf.col('_data_timestamp'), 'HH:mm'))
            .withColumn('Ano', sf.year(sf.col('_data_timestamp')))
            .select('Data', 'Tempo', 'Temperatura', 'Regiao', 'UF', 'Estacao', 'Latitude', 'Longitude', 'Altitude', 'Ano')
        )

        
        print(f"Salvando os dados transformados em {pasta_destino}...")
        df_transformado.write \
            .partitionBy("Ano") \
            .mode("overwrite") \
            .parquet(pasta_destino)

        print("Processamento concluído com sucesso!")

    finally:
        print("Encerrando a sessão Spark.")
        spark.stop()

if __name__ == '__main__':
    main()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/04 14:41:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

+------+---+-------+----------+----------+--------+------------+------------+-------------------+
|Regiao|UF |Estacao|Latitude  |Longitude |Altitude|DataOriginal|HoraOriginal|TemperaturaOriginal|
+------+---+-------+----------+----------+--------+------------+------------+-------------------+
|CO    |DF |NULL   |-15.789444|-47.925835|1159.54 |2011-01-01  |00:00       |19,2               |
|CO    |DF |NULL   |-15.789444|-47.925835|1159.54 |2011-01-01  |01:00       |19,2               |
|CO    |DF |NULL   |-15.789444|-47.925835|1159.54 |2011-01-01  |02:00       |19,1               |
|CO    |DF |NULL   |-15.789444|-47.925835|1159.54 |2011-01-01  |03:00       |18,1               |
|CO    |DF |NULL   |-15.789444|-47.925835|1159.54 |2011-01-01  |04:00       |17,7               |
|CO    |DF |NULL   |-15.789444|-47.925835|1159.54 |2011-01-01  |05:00       |17,7               |
|CO    |DF |NULL   |-15.789444|-47.925835|1159.54 |2011-01-01  |06:00       |17,7               |
|CO    |DF |NULL   |

Processamento concluído com sucesso!
Encerrando a sessão Spark.


In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Leitura INMET com PySpark") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

caminho_parquet = "../data/bronze/INMET_PARQUET/Ano=2011"

df = spark.read.parquet(caminho_parquet)

print("Schema do DataFrame:")
df.printSchema()

print("\nAlgumas linhas do DataFrame:")
df.show()

# Para encerrar a sessão
spark.stop()

Schema do DataFrame:
root
 |-- Data: string (nullable = true)
 |-- Tempo: string (nullable = true)
 |-- Temperatura: float (nullable = true)
 |-- Regiao: string (nullable = true)
 |-- UF: string (nullable = true)
 |-- Estacao: string (nullable = true)
 |-- Latitude: float (nullable = true)
 |-- Longitude: float (nullable = true)
 |-- Altitude: float (nullable = true)


Algumas linhas do DataFrame:


[Stage 1:>                                                          (0 + 1) / 1]

+----------+-----+-----------+------+---+-------+----------+----------+--------+
|      Data|Tempo|Temperatura|Regiao| UF|Estacao|  Latitude| Longitude|Altitude|
+----------+-----+-----------+------+---+-------+----------+----------+--------+
|2011-01-01|00:00|       23.2|     N| AM|   NULL|-2.8963888|-57.758335|    17.0|
|2011-01-01|01:00|       23.2|     N| AM|   NULL|-2.8963888|-57.758335|    17.0|
|2011-01-01|02:00|       23.1|     N| AM|   NULL|-2.8963888|-57.758335|    17.0|
|2011-01-01|03:00|       23.0|     N| AM|   NULL|-2.8963888|-57.758335|    17.0|
|2011-01-01|04:00|       22.8|     N| AM|   NULL|-2.8963888|-57.758335|    17.0|
|2011-01-01|05:00|       22.6|     N| AM|   NULL|-2.8963888|-57.758335|    17.0|
|2011-01-01|06:00|       22.5|     N| AM|   NULL|-2.8963888|-57.758335|    17.0|
|2011-01-01|07:00|       22.3|     N| AM|   NULL|-2.8963888|-57.758335|    17.0|
|2011-01-01|08:00|       22.4|     N| AM|   NULL|-2.8963888|-57.758335|    17.0|
|2011-01-01|09:00|       22.